# ChEBI Multi-label Classification Pipeline

Potok: dane → cechy → DAG → modele → ewaluacja

---
## 0. Setup: importy i logging

In [1]:
import logging
import sys

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

---
## 1. Dane: wczytanie i scaffold split

In [2]:
from chebi_data import load_chebi_dataset, scaffold_split, get_label_columns

DATA_PATH = '../../1_ontology/data/chebi_dataset_train.parquet'
SMILES_COL = 'SMILES'
LABEL_PREFIX = 'class_'

df = load_chebi_dataset(DATA_PATH)
df.head(3)

2026-03-15 00:27:45,065 [INFO] Enabling RDKit 2025.03.6 jupyter extensions
2026-03-15 00:27:45,193 [INFO] Wczytano dane: 33668 próbek, 502 kolumn.


,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# df_train, df_valid = scaffold_split(df, smiles_col=SMILES_COL, train_size=0.8)
#
#
# print(f'Train: {len(df_train):,} próbek')
# print(f'Valid: {len(df_valid):,} próbek')
# print(f'Liczba klas: {n_classes}')

2026-03-14 23:48:17,904 [INFO] Generowanie rusztowań Bemis-Murcko dla scaffold split...


[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:18] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not removing hydrogen atom without neighbors
[23:48:19] WARNING: not r

2026-03-14 23:48:21,079 [INFO] Scaffold split zakończony: train=8000, valid=2000 (proporcja docelowa: 80%)
2026-03-14 23:48:21,081 [INFO] Znaleziono 500 kolumn etykiet (prefix='class_').
Train: 8,000 próbek
Valid: 2,000 próbek
Liczba klas: 500


[23:48:21] WARNING: not removing hydrogen atom without neighbors
[23:48:21] WARNING: not removing hydrogen atom without neighbors


In [3]:
df_train = df

valid_path = "../../1_ontology/model_prep/chebi_dataset_test_empty.parquet"
df_valid = pd.read_parquet(valid_path)

In [4]:
label_cols = get_label_columns(df_train, prefix=LABEL_PREFIX)
n_classes = len(label_cols)

2026-03-15 00:27:51,458 [INFO] Znaleziono 500 kolumn etykiet (prefix='class_').


In [5]:
new_label_data = {col: np.zeros(len(df_valid), dtype=np.int8) for col in label_cols}
df_valid = df_valid.assign(**new_label_data)

C:\Users\domin\AppData\Local\Temp\ipykernel_22900\2679415851.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_valid = df_valid.assign(**new_label_data)
C:\Users\domin\AppData\Local\Temp\ipykernel_22900\2679415851.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_valid = df_valid.assign(**new_label_data)
C:\Users\domin\AppData\Local\Temp\ipykernel_22900\2679415851.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

In [7]:
# Macierze etykiet – oddzielnie dla train i valid
y_train = df_train[label_cols].values.astype(np.float32)
y_valid  = df_valid[label_cols].values.astype(np.float32)

print(f'y_train shape: {y_train.shape}')
print(f'y_valid shape:  {y_valid.shape}')

y_train shape: (33668, 500)
y_valid shape:  (11223, 500)


---
## 2. Inżynieria cech: transformacja train i valid

In [8]:
from chebi_feature_pipeline import ChEBIFeaturePipeline, FeatureConfig

# Konfiguracja – możesz tu wyłączyć niepotrzebne grupy cech:
# np. cfg = FeatureConfig(use_topo_torsion=False, use_klekota_roth=False)
feat_config = FeatureConfig()
feat_pipeline = ChEBIFeaturePipeline(feat_config)

In [9]:
# Train: fit + transform (scaler dopasowywany tylko na train)
X_train = feat_pipeline.fit_transform_train(df_train, smiles_col=SMILES_COL)
print(f'X_train: {X_train.shape}')

2026-03-15 00:28:26,501 [INFO] Budowanie i dopasowywanie potoku cech (zbiór treningowy)...
2026-03-15 00:28:26,502 [INFO] Konfiguracja cech: FeatureConfig(use_maccs=True, use_ecfp_count=True, use_atom_pair=True, use_topo_torsion=True, use_klekota_roth=True, use_rdkit_descs=True, use_domain_features=True, use_smiles_features=True, use_salt_ion_features=True, use_ring_topology=True, use_smarts_patterns=True, use_chain_features=True, use_polymer_features=True, ecfp_radius=2, ecfp_n_bits=2048, atom_pair_count=True, topo_torsion_count=True, n_jobs=-1)
2026-03-15 00:28:26,502 [INFO] [FeatureConfig] MACCS Keys: ON
2026-03-15 00:28:26,503 [INFO] [FeatureConfig] ECFP Count: ON
2026-03-15 00:28:26,504 [INFO] [FeatureConfig] Atom Pair: ON
2026-03-15 00:28:26,505 [INFO] [FeatureConfig] Topological Torsion: ON
2026-03-15 00:28:26,683 [INFO] [FeatureConfig] Klekota-Roth: ON
2026-03-15 00:28:26,684 [INFO] [FeatureConfig] RDKit 2D Descriptors: ON
2026-03-15 00:28:26,685 [INFO] [FeatureConfig] Domain F

[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:26] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] Unusual charge on atom 0 number of radical electrons set to zero
[00:28:27] Unusual charge on atom 0 number of radical electrons set to zero
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00:28:27] WARNING: not removing hydrogen atom without neighbors
[00

2026-03-15 00:32:38,762 [INFO] X_train shape: (33668, 11600)
X_train: (33668, 11600)


In [10]:
# Valid: tylko transform (brak wyciekania danych)
X_valid = feat_pipeline.transform_valid(df_valid, smiles_col=SMILES_COL)
print(f'X_valid: {X_valid.shape}')

2026-03-15 00:32:43,273 [INFO] Transformacja zbioru walidacyjnego (tylko transform)...


[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] Unusual charge on atom 6 number of radical electrons set to zero
[00:32:43] Unusual charge on atom 6 number of radical electrons set to zero
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00:32:43] WARNING: not removing hydrogen atom without neighbors
[00

2026-03-15 00:34:16,341 [INFO] X_valid shape: (11223, 11600)
X_valid: (11223, 11600)


C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


---
## 3. DAG: wczytanie ontologii ChEBI

In [11]:
from chebi_dag import ChEBIDAG

OBO_PATH = 'chebi_classes.txt'

chebi_dag = ChEBIDAG.from_obo(OBO_PATH)
print(f'Węzły: {chebi_dag.graph.number_of_nodes():,}')
print(f'Krawędzie: {chebi_dag.graph.number_of_edges():,}')

2026-03-15 00:34:26,314 [INFO] DAG wczytany z 'chebi_classes.txt': 500 węzłów, 748 krawędzi.
Węzły: 500
Krawędzie: 748


---
## 4. Trening modeli LightGBM

In [13]:
import networkx as nx

# ============================================================
# CELL A — Struktury DAG + weryfikacja
# ============================================================
topo = list(nx.topological_sort(chebi_dag.graph))
topo_filtered = [c for c in topo if c in label_cols]
topo_correct  = list(reversed(topo_filtered))  # korzenie pierwsze, liście ostatnie

col_to_idx     = {col: i for i, col in enumerate(label_cols)}
label_cols_set = set(label_cols)
n_classes      = len(label_cols)

def get_direct_parents(class_name, dag_graph, label_cols_set):
    """Bezposredni rodzice wezla (dag.successors = rodzice, bo krawedz child->parent)."""
    return [p for p in dag_graph.successors(class_name) if p in label_cols_set]

# Sanity check kolejnosci
assert topo_correct[0] == 'class_0', f"Korzen powinien byc pierwszy: {topo_correct[0]}"
pos = {node: i for i, node in enumerate(topo_correct)}
errors = sum(
    1 for node in topo_correct
    for parent in get_direct_parents(node, chebi_dag.graph, label_cols_set)
    if pos[parent] > pos[node]
)
assert errors == 0, f"Bledy kolejnosci topologicznej: {errors}"
print(f"Kolejnosc topologiczna: OK ({len(topo_correct)} klas)")
print(f"X_train : {X_train.shape}")
print(f"X_valid : {X_valid.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_valid  : {y_valid.shape}")


Kolejnosc topologiczna: OK (500 klas)
X_train : (33668, 11600)
X_valid : (11223, 11600)
y_train : (33668, 500)
y_valid  : (11223, 500)


In [14]:
# ============================================================
# CELL B — Konfiguracja LightGBM
# ============================================================
LGBM_PARAMS = {
    "objective":         "binary",
    "metric":            "binary_logloss",
    "boosting_type":     "gbdt",
    "num_leaves":        63,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 20,
    "reg_alpha":         0.1,
    "reg_lambda":        0.1,
    "verbose":           -1,
    "n_jobs":            -1,
}

N_ESTIMATORS = 300   # zwiększ do 500–1000 na finalnym treningu
EARLY_STOP   = 30
THRESHOLD    = 0.5
F1_PERIOD    = 20


In [32]:


# ============================================================
# CELL C — Callback F1
# ============================================================
class F1Callback:
    def __init__(self, X_eval, y_eval_col, period=20, threshold=0.5):
        self.X_eval     = X_eval
        self.y_eval_col = y_eval_col
        self.period     = period
        self.threshold  = threshold

    def __call__(self, env):
        if (env.iteration + 1) % self.period != 0:
            return
        y_prob = env.model.predict(self.X_eval)
        y_pred = (y_prob >= self.threshold).astype(int)
        f1_score(self.y_eval_col, y_pred, average="binary", zero_division=0)


In [36]:
import lightgbm as lgb
import numpy as np
import time
import joblib
import os
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

# Na początku notebooka — globalnie wycisz LightGBM
lgb.register_logger(logging.getLogger("lightgbm"))
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# ============================================================
# CELL D — Trening hierarchiczny
# ============================================================

MODELS_DIR = "models_chebi"
os.makedirs(MODELS_DIR, exist_ok=True)

t0 = time.time()

models            = {}
f1_per_class      = {}

prob_matrix_train = np.zeros((X_train.shape[0], n_classes), dtype=np.float32)
prob_matrix_valid = np.zeros((X_valid.shape[0],  n_classes), dtype=np.float32)

pbar = tqdm(topo_correct, desc="Trening hierarchiczny", unit="klasa")

for class_name in pbar:
    i         = col_to_idx[class_name]
    y_tr_col  = y_train[:, i]
    y_val_col = y_valid[:,  i]

    parent_names = get_direct_parents(class_name, chebi_dag.graph, label_cols_set)
    parent_idx   = [col_to_idx[p] for p in parent_names]

    if parent_idx:
        X_tr_aug  = hstack([X_train, csr_matrix(prob_matrix_train[:, parent_idx])])
        X_val_aug = hstack([X_valid,  csr_matrix(prob_matrix_valid[:,  parent_idx])])
    else:
        X_tr_aug  = X_train
        X_val_aug = X_valid

    n_pos = int(y_tr_col.sum())
    n_neg = len(y_tr_col) - n_pos

    # Klasa stała — brak treningu
    if n_pos == 0 or n_neg == 0:
        const_prob = float(n_pos) / len(y_tr_col)
        prob_matrix_train[:, i] = const_prob
        prob_matrix_valid[:, i] = const_prob

        y_pred_const = (
            np.ones(len(y_tr_col), dtype=np.int8)
            if const_prob >= THRESHOLD
            else np.zeros(len(y_tr_col), dtype=np.int8)
        )
        f1 = f1_score(
            y_tr_col.astype(np.int8),
            y_pred_const,
            average="binary",
            zero_division=1,
        )
        f1_per_class[class_name] = f1
        # Zapisz None jako marker — brak modelu dla tej klasy
        models[class_name] = None
        pbar.set_postfix({
            "klasa":    class_name,
            "rodzice":  len(parent_idx),
            "F1_klasy": f"{f1:.4f} (train/stała)",
            "F1_macro": f"{np.mean(list(f1_per_class.values())):.4f}",
            "drzewa":   0,
        })
        continue

    # Normalny trening
    params_i = {**LGBM_PARAMS, "scale_pos_weight": n_neg / n_pos, "verbose": -1}

    dtrain = lgb.Dataset(X_tr_aug, label=y_tr_col, free_raw_data=False)
    dval   = lgb.Dataset(X_tr_aug, label=y_tr_col,
                         reference=dtrain, free_raw_data=False)

    model = lgb.train(
        params=params_i,
        train_set=dtrain,
        num_boost_round=N_ESTIMATORS,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(EARLY_STOP, verbose=False),
        ],
    )

    # Zapisz model na dysk
    model_path = os.path.join(MODELS_DIR, f"{class_name}.joblib")
    joblib.dump(model, model_path)
    models[class_name] = model_path

    prob_matrix_train[:, i] = model.predict(X_tr_aug).astype(np.float32)
    prob_matrix_valid[:, i] = model.predict(X_val_aug).astype(np.float32)

    f1 = f1_score(
        y_tr_col.astype(np.int8),
        (prob_matrix_train[:, i] >= THRESHOLD).astype(np.int8),
        average="binary", zero_division=0,
    )
    f1_per_class[class_name] = f1

    pbar.set_postfix({
        "klasa":    class_name,
        "rodzice":  len(parent_idx),
        "F1_klasy": f"{f1:.4f} (train)",
        "F1_macro": f"{np.mean(list(f1_per_class.values())):.4f}",
        "drzewa":   model.num_trees(),
    })

pbar.close()
elapsed = time.time() - t0
print(f"\nTrening zakończony: {elapsed/60:.1f} min")
print(f"F1 macro (train, próg {THRESHOLD}): {np.mean(list(f1_per_class.values())):.4f}")
print(f"Modele zapisane w: {MODELS_DIR}/")

Trening hierarchiczny: 100%|██████████| 500/500 [1:22:34<00:00,  9.91s/klasa, klasa=class_138, rodzice=3, F1_klasy=1.0000 (train), F1_macro=0.9933, drzewa=300]


Trening zakończony: 82.6 min
F1 macro (train, próg 0.5): 0.9933
Modele zapisane w: models_chebi/


In [45]:
# ============================================================
# CELL E — Zapis predykcji (submission)
# ============================================================

import pandas as pd
import numpy as np

# Binarne predykcje PRZED propagacją DAG
y_pred_raw = (prob_matrix_valid >= THRESHOLD).astype(np.int8)

# Binarne predykcje PO propagacji DAG
y_pred_dag = chebi_dag.propagate_labels_upward(y_pred_raw, label_cols)

# --- Wspólne kolumny identyfikacyjne ---
id_cols = {}
if "mol_id" in df_valid.columns:
    id_cols["mol_id"] = df_valid["mol_id"].values
if "SMILES" in df_valid.columns:
    id_cols["SMILES"] = df_valid["SMILES"].values

def _add_id_cols(df):
    for col_name, values in reversed(id_cols.items()):
        df.insert(0, col_name, values)
    return df

# 1) Prawdopodobieństwa
df_proba = pd.DataFrame(prob_matrix_valid, columns=label_cols)
df_proba = _add_id_cols(df_proba)
df_proba.to_csv("chebi_submission_proba.csv", index=False)
print(f"Prawdopodobienstwa:   chebi_submission_proba.csv   {df_proba.shape}")

# 2) Binarne PRZED propagacją DAG
df_raw = pd.DataFrame(y_pred_raw, columns=label_cols)
df_raw = _add_id_cols(df_raw)
df_raw.to_csv("chebi_submission_raw.csv", index=False)
print(f"Przed propagacja DAG: chebi_submission_raw.csv     {df_raw.shape}")

# 3) Binarne PO propagacji DAG (submission finalny)
df_dag = pd.DataFrame(y_pred_dag, columns=label_cols)
df_dag = _add_id_cols(df_dag)
df_dag.to_csv("chebi_submission.csv", index=False)
print(f"Po propagacji DAG:    chebi_submission.csv         {df_dag.shape}")

print(f"\nPodglad submission finalnego:")
print(df_dag.head(3))

2026-03-15 02:36:42,283 [INFO] Propagacja DAG: naprawiono 10218 niespójnych etykiet.
Prawdopodobienstwa:   chebi_submission_proba.csv   (11223, 502)
Przed propagacja DAG: chebi_submission_raw.csv     (11223, 502)
Po propagacji DAG:    chebi_submission.csv         (11223, 502)

Podglad submission finalnego:
      mol_id                                             SMILES  class_0  \
0   mol_6861  OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...        1   
1  mol_29793  O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...        1   
2  mol_26953  CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...        1   

   class_1  class_10  class_100  class_101  class_102  class_103  class_104  \
0        1         0          0          1          0          0          0   
1        1         1          0          0          0          0          0   
2        1         1          0          0          0          0          0   

   ...  class_90  class_91  class_92  class_93  class_94  class_95  cl

In [41]:
sample_submission = pd.read_parquet("../../1_ontology/model_prep/chebi_submission_example.parquet")

In [42]:
sample_submission

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...,0,0,1,0,0,0,0,1,...,1,1,0,0,0,1,1,0,0,0
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...,1,1,1,0,0,0,0,1,...,0,1,0,1,1,0,0,0,0,1
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,1,0,0,1,1,0,1,1,...,0,0,0,1,0,1,1,1,1,0
3,mol_26053,C=C1CCC(C(C)C)CC1,1,1,0,1,1,0,0,0,...,1,1,1,1,0,1,1,1,1,0
4,mol_18653,O=C(O)C1CCCCC1=O,0,0,0,0,1,0,0,1,...,0,1,0,1,1,1,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11218,mol_13480,Oc1cc2cc[nH]c2cc1O,0,0,0,1,0,0,1,0,...,1,0,0,0,0,1,0,0,0,0
11219,mol_42067,CNCC(=O)OCc1cccnc1N(C)C(=O)OC(C)[n+]1cnn(C[C@]...,1,1,0,0,0,0,0,1,...,0,0,0,1,0,0,1,1,1,0
11220,mol_43978,N[C@@H](CCC(=O)[O-])C(O)=Nc1ccc2ccccc2c1,1,1,1,0,0,1,0,1,...,1,1,1,1,1,0,1,0,1,1
11221,mol_16268,COc1cc(-c2[o+]c3cc(O)cc(O)c3cc2O[C@@H]2O[C@H](...,0,1,1,0,0,1,0,0,...,0,1,1,0,1,0,1,0,0,1


In [50]:
# Dopasuj kolumny do sample_submission
final_cols = sample_submission.columns.tolist()  # ['mol_id', 'class_0', ..., 'class_499']

# 2) Przed propagacją DAG
df_raw = pd.DataFrame(y_pred_raw, columns=label_cols)
df_raw.insert(0, "mol_id", df_valid["mol_id"].values)
df_raw.insert(1, "SMILES", df_valid["SMILES"].values)
df_raw = df_raw[final_cols]  # upewnij się że kolejność kolumn jest identyczna
df_raw.to_parquet("task_2_submission_raw.parquet", index=False)


In [61]:
# 2) Przed propagacją DAG
df_raw = pd.DataFrame(prob_matrix_valid, columns=label_cols)
df_raw.insert(0, "mol_id", df_valid["mol_id"].values)
df_raw.insert(1, "SMILES", df_valid["SMILES"].values)
df_raw = df_raw[final_cols]  # upewnij się że kolejność kolumn jest identyczna
df_raw.to_parquet("task_2_submission_proba.parquet", index=False)

In [55]:
# ============================================================
# CELL E — Optymalizacja progu per klasa na trainsecie
# ============================================================

print("Szukam optymalnych progów na trainsecie...")

optimal_thresholds = {}
for i, class_name in enumerate(tqdm(label_cols, desc="Progi", leave=False)):
    y_true = y_train[:, i].astype(np.int8)
    y_prob = prob_matrix_train[:, i]

    best_f1, best_thr = 0.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.05):
        f1 = f1_score(y_true, (y_prob >= thr).astype(int),
                      average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    optimal_thresholds[class_name] = best_thr

# Zapisz progi do pliku — przyda się przy ponownym użyciu modeli
import json
with open("optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f)
print("Progi zapisane: optimal_thresholds.json")

# Predykcja z optymalnymi progami
y_pred_raw = np.zeros((X_valid.shape[0], n_classes), dtype=np.int8)
for i, class_name in enumerate(label_cols):
    y_pred_raw[:, i] = (
        prob_matrix_valid[:, i] >= optimal_thresholds[class_name]
    ).astype(np.int8)

Szukam optymalnych progów na trainsecie...


Progi zapisane: optimal_thresholds.json


In [57]:
# 2) Przed propagacją DAG
df_raw = pd.DataFrame(y_pred_raw, columns=label_cols)
df_raw.insert(0, "mol_id", df_valid["mol_id"].values)
df_raw.insert(1, "SMILES", df_valid["SMILES"].values)
df_raw = df_raw[final_cols]  # upewnij się że kolejność kolumn jest identyczna
df_raw.to_parquet("task_2_submission_optimal_thresholds.parquet", index=False)

In [56]:
y_pred_raw

array([[1, 1, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int8)

In [51]:
# 3) Po propagacji DAG
df_dag = pd.DataFrame(y_pred_dag, columns=label_cols)
df_dag.insert(0, "mol_id", df_valid["mol_id"].values)
df_dag.insert(1, "SMILES", df_valid["SMILES"].values)
df_dag = df_dag[final_cols]
df_dag.to_parquet("task_2_submission_dag.parquet", index=False)  # parquet jeśli konkurs tego wymaga